In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
import os

# Add src to path
sys.path.append('..')
from src.data_processor import DataProcessor

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
%matplotlib inline

print("✓ Libraries loaded successfully")

## 1. Load and Inspect Data

In [ ]:
# Initialize data processor
processor = DataProcessor()

# Option 1: Load real Spotify data (download from Kaggle first)
# processor.load_data('../data/raw/spotify_data.csv')

# Option 2: Create synthetic data for demonstration
df = processor.create_sample_data(n_samples=5000)

print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Basic info
print("=" * 50)
print("DATASET INFO")
print("=" * 50)
print(f"\nTotal tracks: {len(df):,}")
print(f"Features: {len(df.columns)}")
print(f"\nColumn types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Statistical summary
df.describe().round(2)

## 2. Audio Feature Distributions

In [ ]:
# Audio features to analyze
audio_features = ['danceability', 'energy', 'loudness', 'speechiness', 
                  'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

available_features = [f for f in audio_features if f in df.columns]

# Distribution plots
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, feature in enumerate(available_features):
    ax = axes[i]
    sns.histplot(df[feature], kde=True, ax=ax, color='steelblue')
    ax.set_title(f'{feature.capitalize()} Distribution')
    ax.set_xlabel(feature)
    
plt.tight_layout()
plt.savefig('../data/processed/feature_distributions.png', dpi=150)
plt.show()

In [ ]:
# Interactive distribution with Plotly
fig = make_subplots(rows=3, cols=3, subplot_titles=available_features)

for i, feature in enumerate(available_features):
    row = i // 3 + 1
    col = i % 3 + 1
    fig.add_trace(
        go.Histogram(x=df[feature], name=feature, nbinsx=50),
        row=row, col=col
    )
    
fig.update_layout(height=800, title_text="Audio Feature Distributions", showlegend=False)
fig.show()

## 3. Correlation Analysis

In [ ]:
# Correlation matrix
correlation_features = available_features + ['popularity'] if 'popularity' in df.columns else available_features
corr_matrix = df[correlation_features].corr()

# Heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='RdYlBu_r', center=0, square=True,
            linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('../data/processed/correlation_matrix.png', dpi=150)
plt.show()

In [ ]:
# Key correlations with popularity
if 'popularity' in df.columns:
    popularity_corr = corr_matrix['popularity'].drop('popularity').sort_values(ascending=False)
    
    plt.figure(figsize=(10, 6))
    colors = ['green' if x > 0 else 'red' for x in popularity_corr.values]
    popularity_corr.plot(kind='barh', color=colors)
    plt.xlabel('Correlation with Popularity')
    plt.title('Feature Correlation with Popularity')
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    plt.tight_layout()
    plt.show()
    
    print("\nTop positive correlations:")
    print(popularity_corr.head(3))
    print("\nTop negative correlations:")
    print(popularity_corr.tail(3))

## 4. Popularity Analysis

In [ ]:
if 'popularity' in df.columns:
    # Popularity distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(df['popularity'], bins=50, color='steelblue', edgecolor='white')
    axes[0].set_xlabel('Popularity Score')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Popularity Distribution')
    axes[0].axvline(df['popularity'].mean(), color='red', linestyle='--', label=f'Mean: {df["popularity"].mean():.1f}')
    axes[0].axvline(df['popularity'].median(), color='green', linestyle='--', label=f'Median: {df["popularity"].median():.1f}')
    axes[0].legend()
    
    # Box plot
    axes[1].boxplot(df['popularity'])
    axes[1].set_ylabel('Popularity Score')
    axes[1].set_title('Popularity Box Plot')
    
    plt.tight_layout()
    plt.show()
    
    # Stats
    print(f"\nPopularity Statistics:")
    print(f"Mean: {df['popularity'].mean():.2f}")
    print(f"Median: {df['popularity'].median():.2f}")
    print(f"Std: {df['popularity'].std():.2f}")
    print(f"Range: {df['popularity'].min()} - {df['popularity'].max()}")

In [ ]:
# Popularity categories
if 'popularity' in df.columns:
    df['popularity_category'] = pd.cut(df['popularity'], 
                                        bins=[0, 25, 50, 75, 100],
                                        labels=['Low', 'Medium', 'High', 'Very High'])
    
    # Count by category
    category_counts = df['popularity_category'].value_counts()
    
    fig = px.pie(values=category_counts.values, names=category_counts.index,
                 title='Songs by Popularity Category',
                 color_discrete_sequence=px.colors.sequential.Viridis)
    fig.show()

## 5. Feature Relationships

In [ ]:
# Energy vs Valence (Mood Quadrant)
if 'energy' in df.columns and 'valence' in df.columns:
    fig = px.scatter(df.sample(min(2000, len(df))), 
                     x='valence', y='energy',
                     color='popularity' if 'popularity' in df.columns else None,
                     opacity=0.6,
                     title='Energy vs Valence (Mood Quadrant)',
                     labels={'valence': 'Valence (Positivity)', 'energy': 'Energy'},
                     color_continuous_scale='Viridis')
    
    # Add quadrant labels
    fig.add_annotation(x=0.25, y=0.75, text="Angry/Turbulent", showarrow=False, font=dict(size=12))
    fig.add_annotation(x=0.75, y=0.75, text="Happy/Energetic", showarrow=False, font=dict(size=12))
    fig.add_annotation(x=0.25, y=0.25, text="Sad/Depressed", showarrow=False, font=dict(size=12))
    fig.add_annotation(x=0.75, y=0.25, text="Calm/Peaceful", showarrow=False, font=dict(size=12))
    
    # Add quadrant lines
    fig.add_hline(y=0.5, line_dash="dash", line_color="gray")
    fig.add_vline(x=0.5, line_dash="dash", line_color="gray")
    
    fig.show()

In [ ]:
# Danceability vs Tempo
if 'danceability' in df.columns and 'tempo' in df.columns:
    fig = px.scatter(df.sample(min(2000, len(df))),
                     x='tempo', y='danceability',
                     color='energy' if 'energy' in df.columns else None,
                     opacity=0.6,
                     title='Danceability vs Tempo',
                     color_continuous_scale='Plasma')
    fig.show()

In [ ]:
# Pair plot for key features
key_features = ['energy', 'valence', 'danceability', 'acousticness']
available_key = [f for f in key_features if f in df.columns]

if available_key and 'popularity' in df.columns:
    sample_df = df[available_key + ['popularity_category']].sample(min(1000, len(df)))
    
    g = sns.pairplot(sample_df, hue='popularity_category', 
                     palette='viridis', diag_kind='kde',
                     plot_kws={'alpha': 0.6})
    g.fig.suptitle('Pair Plot of Key Audio Features', y=1.02)
    plt.show()

## 6. Data Quality Summary

In [ ]:
# Data quality report
print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)

print(f"\n📊 Dataset Size: {len(df):,} tracks")
print(f"📋 Features: {len(df.columns)}")

# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(f"\n❌ Missing Values:")
if missing.sum() > 0:
    for col, pct in missing_pct[missing_pct > 0].items():
        print(f"   - {col}: {pct}%")
else:
    print("   No missing values! ✓")

# Duplicates
duplicates = df.duplicated().sum()
print(f"\n📋 Duplicates: {duplicates:,} ({duplicates/len(df)*100:.2f}%)")

# Feature ranges
print(f"\n📐 Feature Ranges (0-1 expected):")
normalized_features = ['danceability', 'energy', 'speechiness', 'acousticness', 
                       'instrumentalness', 'liveness', 'valence']
for feature in normalized_features:
    if feature in df.columns:
        min_val, max_val = df[feature].min(), df[feature].max()
        status = "✓" if 0 <= min_val and max_val <= 1 else "⚠"
        print(f"   {status} {feature}: [{min_val:.3f}, {max_val:.3f}]")

In [ ]:
# Save cleaned data
processor.raw_data = df
processor.clean_data()

# Save processed data
output_path = '../data/processed/cleaned_music_data.csv'
processor.save_processed_data(output_path)

print(f"\n✓ Cleaned data saved to {output_path}")

## 7. Key Insights Summary

### Findings:
1. **Feature Distributions**: Most audio features follow expected distributions
2. **Correlations**: Energy and loudness show strong positive correlation
3. **Popularity**: [Add findings based on your data]
4. **Mood Quadrant**: Energy-Valence space effectively separates different mood categories

### Next Steps:
- Proceed to clustering analysis (02_clustering_analysis.ipynb)
- Use identified feature relationships for emotion mapping